# Evaluación completa de la validación cruzada de 5 folds (fold 0 recalculado y folds 2-4) con análisis pareado

El notebook realiza los siguientes pasos:

1. **Recálculo del fold 0** con las mismas funciones de métricas que el fold 1 (incluidas `topo_precision` y `topo_sensitivity`, que la evaluación original del fold 0 no calculaba). De este modo, las seis métricas están disponibles para **n = 1000** y no solo para 800.
2. **Evaluación de los folds 2, 3 y 4** (baseline y v1), con descarga de predicciones y GT mediante la API de Drive (sin FUSE) y cálculo de métricas caso a caso con checkpoint.
3. **Reutilización del fold 1** sin recalcular, ya que contiene las seis métricas completas de la evaluación del notebook 32.
4. Con los cinco folds homogéneos, **tres análisis complementarios** (y no solo un p-valor):
   - **(A) Resultado pooled:** las 1000 imágenes conjuntas, con Wilcoxon pareado y delta medio y mediano.
   - **(B) Resultado por fold:** cada uno de los cinco folds evaluado de forma independiente (su propio Wilcoxon y su propio delta).
   - **(C) Consistencia del efecto entre folds:** además de la media ± SD de cada métrica entre folds, la media ± SD del **delta (v1 - baseline)** entre los cinco folds, que responde directamente a si el efecto de soft-clDice es reproducible o específico de un fold.

**Convenio de signo (unificado en todo el notebook):**

> **delta > 0 indica siempre que v1 mejora respecto al baseline**, para las seis métricas:
> - `delta_dice = dice_v1 - dice_baseline`
> - `delta_cldice = cldice_v1 - cldice_baseline`
> - `delta_topo_precision = topo_precision_v1 - topo_precision_baseline`
> - `delta_topo_sensitivity = topo_sensitivity_v1 - topo_sensitivity_baseline`
> - `delta_n_componentes = n_componentes_baseline - n_componentes_v1` (menos componentes es mejor, por lo que el orden se invierte)
> - `delta_hd95 = hd95_baseline - hd95_v1` (un HD95 menor es mejor, por lo que el orden también se invierte)

**Runtime:** solo CPU (sin acelerador); no requiere GPU.


## 1. Autenticación con la API de Drive

Drive se monta también mediante FUSE, pero únicamente para operaciones ligeras (lectura y escritura de CSV pequeños) y nunca para las imágenes de evaluación.


In [1]:
from google.colab import auth
auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

creds, _ = google.auth.default()
drive_api = build("drive", "v3", credentials=creds)
print("Api de Drive autenticada.")

from google.colab import drive
drive.mount("/content/drive", force_remount=True)
print("Drive montado (solo para CSVs pequenos, no para las imagenes).")

import socket
socket.setdefaulttimeout(90)
print("Timeout de socket (90s) activado para las descargas.")


Api de Drive autenticada.
Mounted at /content/drive
Drive montado (solo para CSVs pequenos, no para las imagenes).
Timeout de socket (90s) activado para las descargas.


## 2. Instalación de dependencias de evaluación

En un runtime nuevo sin acelerador, estas librerías no están preinstaladas y son necesarias para el cálculo de métricas (sección 6).


In [2]:
!pip install -q medpy scikit-image SimpleITK
print("Dependencias instaladas.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 12.8 MB/s eta 0:00:00
Dependencias instaladas.


## 3. Configuración


In [3]:
import os, io, time, shutil
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")

DATASET_FOLDER_NAME = "Dataset503_ImageCAS1000"
CONFIGURATION = "3d_fullres"

BASELINE_TRAINER = "nnUNetTrainer_100epochs"
V1_TRAINER = "nnUNetTrainerV1_1000casos_100epochs"

# El fold 0 se recalcula aquí (para disponer de topo_precision y topo_sensitivity en
# las 1000 imágenes); los folds 2-4 se calculan por primera vez en este notebook;
# el fold 1 ya está completo (notebook 32) y solo se carga.
FOLDS_A_PROCESAR = [0, 2, 3, 4]
FOLD_YA_COMPLETO = 1
TODOS_LOS_FOLDS = [0, 1, 2, 3, 4]

LOCAL_EVAL_ROOT = Path("/content/eval_folds")

print("BASE (Drive, solo CSVs):", BASE)
print("Folds a procesar en este notebook:", FOLDS_A_PROCESAR)
print("Fold reutilizado sin recalcular:", FOLD_YA_COMPLETO)


BASE (Drive, solo CSVs): /content/drive/MyDrive/TFM_nnUNet
Folds a procesar en este notebook: [0, 2, 3, 4]
Fold reutilizado sin recalcular: 1


## 4. Funciones de navegación y descarga de Drive mediante API (sin FUSE)


In [4]:
def buscar_id(nombre, parent_id, carpeta=True):
    mime = " and mimeType = 'application/vnd.google-apps.folder'" if carpeta else ""
    q = f"name = '{nombre}' and '{parent_id}' in parents and trashed = false{mime}"
    res = drive_api.files().list(q=q, fields="files(id, name)", pageSize=5).execute()
    archivos = res.get("files", [])
    if not archivos:
        raise FileNotFoundError(f"No se encontro '{nombre}' dentro de {parent_id}")
    return archivos[0]["id"]

def listar_hijos(parent_id):
    archivos = []
    page_token = None
    while True:
        res = drive_api.files().list(
            q=f"'{parent_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, size, mimeType)",
            pageSize=1000,
            pageToken=page_token,
        ).execute()
        archivos.extend(res.get("files", []))
        page_token = res.get("nextPageToken")
        if not page_token:
            break
    return archivos

def descargar_lista(archivos_remotos, carpeta_local, max_reintentos=5):
    carpeta_local.mkdir(parents=True, exist_ok=True)
    pendientes = list(archivos_remotos)
    intento = 0
    while pendientes and intento < max_reintentos:
        intento += 1
        fallidos = []
        for i, meta in enumerate(pendientes):
            destino = carpeta_local / meta["name"]
            tam_esperado = int(meta.get("size", 0) or 0)
            try:
                if destino.exists() and (tam_esperado == 0 or destino.stat().st_size == tam_esperado):
                    continue
                request = drive_api.files().get_media(fileId=meta["id"])
                fh = io.FileIO(destino, "wb")
                downloader = MediaIoBaseDownload(fh, request, chunksize=1024 * 1024 * 20)
                done = False
                while not done:
                    status, done = downloader.next_chunk(num_retries=3)
                fh.close()
            except Exception as e:
                fallidos.append(meta)
            if i % 50 == 0:
                print(f"    {i}/{len(pendientes)} procesados ({len(fallidos)} fallos hasta ahora)")
        pendientes = fallidos
        if pendientes:
            print(f"    Quedan {len(pendientes)} fallidos, esperando 15s...")
            time.sleep(15)
    if pendientes:
        print(f"    ATENCION: siguen fallando {len(pendientes)} archivos tras {max_reintentos} intentos.")
    return pendientes

print("Funciones de Drive API listas (descarga secuencial, de referencia).")


Funciones de Drive API listas (descarga secuencial, de referencia).


## 5. Descarga paralela (la misma del notebook 33, aceleración medida ×6.1)

`descargar_lista` (secuencial) aporta una mejora limitada: la descarga de ≈ 600 archivos por fold tiene su propia latencia de API, pero **no constituyó el cuello de botella** de las ≈ 6 h que requirió el fold 0, que correspondió al **cálculo de métricas** (sección 8, `evaluar_trainer_fold`). Este cálculo se paraleliza más adelante con procesos (y no con threads, ya que es CPU-bound: skeletonize y HD95 por caso, sin espera de red). La descarga se mantiene en paralelo (8 workers, cliente de Drive por hilo, escritura a `.part`, verificación por tamaño, renombrado atómico y backoff exponencial; aceleración ×6.1 medida en el notebook 33) porque no tiene coste adicional y reduce igualmente el tiempo total.


In [5]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

_thread_local = threading.local()

def _drive_service_del_hilo():
    if not hasattr(_thread_local, "service"):
        _thread_local.service = build("drive", "v3", credentials=creds)
    return _thread_local.service

def _descargar_un_archivo(meta, carpeta_local, max_reintentos=5):
    destino = carpeta_local / meta["name"]
    tam_esperado = int(meta.get("size", 0) or 0)
    if destino.exists() and (tam_esperado == 0 or destino.stat().st_size == tam_esperado):
        return ("skip", meta["name"], None)
    parcial = destino.with_name(destino.name + ".part")
    espera = 5
    for intento in range(1, max_reintentos + 1):
        try:
            service = _drive_service_del_hilo()
            request = service.files().get_media(fileId=meta["id"])
            fh = io.FileIO(parcial, "wb")
            downloader = MediaIoBaseDownload(fh, request, chunksize=1024 * 1024 * 20)
            done = False
            while not done:
                status, done = downloader.next_chunk(num_retries=3)
            fh.close()
            if tam_esperado and parcial.stat().st_size != tam_esperado:
                raise IOError(f"Tamano incorrecto: {parcial.stat().st_size} != {tam_esperado}")
            parcial.replace(destino)
            return ("ok", meta["name"], None)
        except Exception as e:
            if parcial.exists():
                parcial.unlink(missing_ok=True)
            if intento == max_reintentos:
                return ("fail", meta["name"], f"{type(e).__name__}: {e}")
            time.sleep(espera)
            espera = min(espera * 2, 60)

def descargar_lista_paralelo(archivos_remotos, carpeta_local, n_workers=8, max_reintentos=5):
    carpeta_local.mkdir(parents=True, exist_ok=True)
    fallidos = []
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futuros = {
            executor.submit(_descargar_un_archivo, meta, carpeta_local, max_reintentos): meta
            for meta in archivos_remotos
        }
        completados = 0
        for futuro in as_completed(futuros):
            estado, nombre, error = futuro.result()
            completados += 1
            if estado == "fail":
                fallidos.append(futuros[futuro])
            if completados % 50 == 0 or completados == len(archivos_remotos):
                print(f"    {completados}/{len(archivos_remotos)} procesados "
                      f"({len(fallidos)} fallos hasta ahora)")
    dt = time.time() - t0
    print(f"    Descarga paralela ({n_workers} workers): {len(archivos_remotos)} archivos en "
          f"{dt/60:.1f} min ({dt/max(len(archivos_remotos),1):.2f} s/archivo)")
    return fallidos, dt

print("Funcion de descarga paralela lista (8 workers por defecto).")


Funcion de descarga paralela lista (8 workers por defecto).


## 6. Funciones de métricas (idénticas a las de los notebooks 28, 29 y 32 para garantizar la comparabilidad)


In [6]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage
from skimage.morphology import skeletonize
from medpy.metric.binary import hd95 as medpy_hd95

CONECTIVIDAD_26 = np.ones((3, 3, 3), dtype=int)

def cargar_volumen(path):
    img = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(img).astype(np.uint8)
    spacing = img.GetSpacing()[::-1]
    return array, spacing

def dice_score(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    return (2.0 * inter) / (pred.sum() + gt.sum() + 1e-8)

def metricas_topologicas(pred, gt):
    skel_pred = skeletonize(pred.astype(bool))
    skel_gt = skeletonize(gt.astype(bool))
    tprec = (skel_pred & gt).sum() / (skel_pred.sum() + 1e-8)
    tsens = (skel_gt & pred).sum() / (skel_gt.sum() + 1e-8)
    cldice = (2 * tprec * tsens) / (tprec + tsens + 1e-8)
    return cldice, tprec, tsens

def componentes_conexas(pred):
    labeled, n = ndimage.label(pred, structure=CONECTIVIDAD_26)
    return labeled, n

def cobertura_componente_principal(pred, gt):
    labeled, n = componentes_conexas(pred)
    if n == 0:
        return 0.0, 0
    tamanos = ndimage.sum(pred, labeled, index=range(1, n + 1))
    idx_mayor = np.argmax(tamanos) + 1
    componente_mayor = labeled == idx_mayor
    cobertura = (componente_mayor & gt.astype(bool)).sum() / (gt.sum() + 1e-8)
    return cobertura, n

def hd95_seguro(pred, gt, spacing):
    if pred.sum() == 0 or gt.sum() == 0:
        return np.nan
    return medpy_hd95(pred, gt, voxelspacing=spacing)

def calcular_metricas_caso(pred_path, gt_path):
    pred, spacing = cargar_volumen(pred_path)
    gt, _ = cargar_volumen(gt_path)
    cobertura, n_componentes = cobertura_componente_principal(pred, gt)
    cldice, topo_precision, topo_sensitivity = metricas_topologicas(pred, gt)
    return {
        "dice": dice_score(pred, gt),
        "cldice": cldice,
        "topo_precision": topo_precision,
        "topo_sensitivity": topo_sensitivity,
        "cobertura_principal": cobertura,
        "n_componentes": n_componentes,
        "hd95_crudo_mm": hd95_seguro(pred, gt, spacing),
    }

print("Funciones de metricas cargadas.")


Funciones de metricas cargadas.


## 7. Convenio de signo único

`delta_mejora(metrica, v_base, v_v1) > 0` indica siempre que v1 mejora. Se emplea en el análisis por fold, en el pooled y en la tabla de consistencia, de modo que no sea necesario recordar que HD95 y número de componentes se interpretan en sentido inverso a Dice y clDice.


In [7]:
# (columna, nombre legible, orientación): orientación = +1 si «más es mejor»
# (Dice, clDice, topo_precision, topo_sensitivity) y -1 si «menos es mejor»
# (n_componentes, HD95).
METRICAS = [
    ("dice", "Dice", +1),
    ("cldice", "clDice", +1),
    ("topo_precision", "Topological precision", +1),
    ("topo_sensitivity", "Topological sensitivity", +1),
    ("n_componentes", "n_componentes", -1),
    ("hd95_crudo_mm", "HD95", -1),
]

def delta_mejora(col, orientacion, v_base, v_v1):
    """positivo = v1 mejora, para cualquier metrica."""
    return orientacion * (v_v1 - v_base)

print("Convenio de signo definido:", [m[0] for m in METRICAS])


Convenio de signo definido: ['dice', 'cldice', 'topo_precision', 'topo_sensitivity', 'n_componentes', 'hd95_crudo_mm']


## 8. `localizar_y_descargar_fold` y `evaluar_trainer_fold`: descarga y evaluación con checkpoint, parametrizadas por fold

Cada fold utiliza sus propias carpetas locales (`/content/eval_folds/fold_X/...`) y su propio checkpoint en Drive (`checkpoint_foldX_<trainer>.csv`). El procedimiento es el mismo para el fold 0 (recálculo) y para los folds 2-4 (cálculo nuevo).

El cálculo de métricas emplea `ProcessPoolExecutor` (procesos reales, no threads, por ser CPU-bound: skeletonize y HD95 por caso). Los workers solo calculan; el proceso principal es el único que escribe el checkpoint en Drive (cada 10 casos), para evitar que ocho procesos escriban el mismo CSV simultáneamente. Si el checkpoint ya contiene casos calculados, solo se reparten entre los workers los que faltan, de modo que no se recalcula ni se pierde nada de lo ya calculado.


In [8]:
from concurrent.futures import ProcessPoolExecutor

def localizar_y_descargar_fold(fold):
    id_tfm = buscar_id("TFM_nnUNet", "root")
    id_results = buscar_id("nnUNet_results", id_tfm)
    id_dataset_res = buscar_id(DATASET_FOLDER_NAME, id_results)

    local_val = {}
    archivos_val_por_trainer = {}
    for trainer in (BASELINE_TRAINER, V1_TRAINER):
        id_trainer_cfg = buscar_id(f"{trainer}__nnUNetPlans__{CONFIGURATION}", id_dataset_res)
        id_fold = buscar_id(f"fold_{fold}", id_trainer_cfg)
        id_val = buscar_id("validation", id_fold)
        archivos_val = listar_hijos(id_val)
        archivos_val_nii = [f for f in archivos_val if f["name"].endswith(".nii.gz")]
        archivos_val_por_trainer[trainer] = archivos_val_nii
        assert len(archivos_val_nii) == 200, (
            f"fold {fold} / {trainer}: {len(archivos_val_nii)} predicciones, se esperaban 200."
        )
        local_val[trainer] = LOCAL_EVAL_ROOT / f"fold_{fold}" / f"val_{trainer}"

    id_prep = buscar_id("nnUNet_preprocessed", id_tfm)
    id_dataset_prep = buscar_id(DATASET_FOLDER_NAME, id_prep)
    id_gt = buscar_id("gt_segmentations", id_dataset_prep)
    archivos_gt_todos = listar_hijos(id_gt)

    nombres_val = {f["name"] for f in next(iter(archivos_val_por_trainer.values()))}
    archivos_gt_val = [f for f in archivos_gt_todos if f["name"] in nombres_val]
    local_gt = LOCAL_EVAL_ROOT / f"fold_{fold}" / "gt_segmentations"

    print(f"  fold {fold}: descargando {len(archivos_val_por_trainer[BASELINE_TRAINER])*2} predicciones "
          f"+ {len(archivos_gt_val)} GT (en paralelo, 8 workers)...")
    for trainer, archivos in archivos_val_por_trainer.items():
        pend, _dt = descargar_lista_paralelo(archivos, local_val[trainer], n_workers=8)
        assert not pend, f"fold {fold} / {trainer}: quedaron predicciones sin descargar: {pend}"
    pend_gt, _dt = descargar_lista_paralelo(archivos_gt_val, local_gt, n_workers=8)
    assert not pend_gt, f"fold {fold}: quedaron GT sin descargar: {pend_gt}"

    return local_val, local_gt


N_WORKERS_METRICAS = min(8, os.cpu_count() or 4)


def calcular_metricas_worker(tarea):
    """Se ejecuta en un proceso hijo (ProcessPoolExecutor). Solo calcula --
    no toca Drive ni el checkpoint, para que 8 procesos no escriban a la vez.
    """
    pred_path, gt_path = tarea
    case_id = pred_path.name.replace(".nii.gz", "")
    try:
        m = calcular_metricas_caso(pred_path, gt_path)
        m["case_id"] = case_id
        return m
    except Exception as e:
        return {"case_id": case_id, "_error": f"{type(e).__name__}: {e}"}


def evaluar_trainer_fold(fold, trainer_name, local_val, local_gt, n_workers=N_WORKERS_METRICAS, guardar_cada=10):
    """Calculo de metricas caso a caso PARALELIZADO con multiprocessing real
    (ProcessPoolExecutor, no threads -- esto es CPU-bound: skeletonize + HD95
    por caso). Cada worker solo calcula; el proceso principal es el UNICO que
    escribe el checkpoint en Drive, cada `guardar_cada` casos (para no tener
    varios procesos escribiendo el mismo CSV a la vez).

    Reanudacion: si ya hay un checkpoint valido (con columnas topologicas),
    solo se reparten entre los workers los case_id que faltan -- no se
    recalcula nada de lo que ya esta guardado (asi no se pierden las horas ya
    invertidas en fold 0 y fold 1).
    """
    carpeta_val = local_val[trainer_name]
    preds = sorted(carpeta_val.glob("*.nii.gz"))
    if not preds:
        raise RuntimeError(f"No se encontraron predicciones en {carpeta_val}")

    checkpoint_path = BASE / f"checkpoint_fold{fold}_{trainer_name}.csv"

    filas = []
    ya_hechos = set()
    if checkpoint_path.exists():
        df_prev = pd.read_csv(checkpoint_path)
                # Si el checkpoint procede de una versión antigua sin las columnas topológicas,
                # se descarta y se recalcula el fold completo (para no mezclar formatos).
        if {"topo_precision", "topo_sensitivity"}.issubset(df_prev.columns):
            filas = df_prev.to_dict("records")
            ya_hechos = set(df_prev["case_id"])
            print(f"    Reanudando fold {fold}/{trainer_name}: {len(ya_hechos)} casos ya calculados.")
        else:
            print(f"    Checkpoint antiguo de fold {fold}/{trainer_name} sin columnas topologicas -- se recalcula desde cero.")

    total = len(preds)
    tareas = []
    for pred_path in preds:
        case_id = pred_path.name.replace(".nii.gz", "")
        if case_id in ya_hechos:
            continue
        gt_path = local_gt / pred_path.name
        if gt_path.exists():
            tareas.append((pred_path, gt_path))

    if not tareas:
        print(f"    [fold {fold} / {trainer_name}] nada pendiente, los {total} casos ya estaban calculados.")
    else:
        print(f"    [fold {fold} / {trainer_name}] calculando {len(tareas)} casos pendientes "
              f"con {n_workers} procesos en paralelo...")
        completados_sin_guardar = 0
        with ProcessPoolExecutor(max_workers=n_workers) as executor:
            futuros = [executor.submit(calcular_metricas_worker, t) for t in tareas]
            for futuro in as_completed(futuros):
                resultado = futuro.result()
                if "_error" in resultado:
                    print(f"      ATENCION: fallo en {resultado['case_id']}: {resultado['_error']}")
                    continue
                filas.append(resultado)
                completados_sin_guardar += 1
                if completados_sin_guardar >= guardar_cada:
                    pd.DataFrame(filas).to_csv(checkpoint_path, index=False)
                    completados_sin_guardar = 0
                    print(f"    [fold {fold} / {trainer_name}] {len(filas)}/{total} procesados -- checkpoint guardado")
        pd.DataFrame(filas).to_csv(checkpoint_path, index=False)
        print(f"    [fold {fold} / {trainer_name}] {len(filas)}/{total} -- checkpoint final guardado")

    df = pd.DataFrame(filas)
    resumen = {
        "dice_mean": df["dice"].mean(),
        "cldice_mean": df["cldice"].mean(),
        "topo_precision_mean": df["topo_precision"].mean(),
        "topo_sensitivity_mean": df["topo_sensitivity"].mean(),
        "n_componentes_mean": df["n_componentes"].mean(),
        "hd95_crudo_mean": df["hd95_crudo_mm"].mean(skipna=True),
        "n_casos": len(df),
    }
    return df, resumen

print(f"Funciones de evaluacion por fold listas (calculo de metricas con {N_WORKERS_METRICAS} procesos en paralelo).")


Funciones de evaluacion por fold listas (calculo de metricas con 2 procesos en paralelo).


## 9. Test pareado y resumen de un fold (función reutilizable, con el convenio de signo único)

Devuelve el dataframe caso a caso (con todos los `delta_*` en el convenio «positivo = mejora con v1») y un diccionario con el delta medio y el p-valor de Wilcoxon por métrica, para construir posteriormente las tablas B y C.


In [9]:
from scipy import stats

def comparar_fold(df_base, df_v1, resumen_base, resumen_v1, fold, imprimir_tabla_completa=False):
    df_base_s = df_base.sort_values("case_id").reset_index(drop=True)
    df_v1_s = df_v1.sort_values("case_id").reset_index(drop=True)
    assert list(df_base_s["case_id"]) == list(df_v1_s["case_id"]), \
        f"fold {fold}: case_id no coinciden entre baseline y v1."

    comp = df_base_s.merge(df_v1_s, on="case_id", suffixes=("_baseline", "_v1"))
    comp["fold"] = fold

    for col, _, orientacion in METRICAS:
        comp[f"delta_{col}"] = orientacion * (comp[f"{col}_v1"] - comp[f"{col}_baseline"])

    print(f"\n=== fold {fold}: resumen ===")
    print(f"Dice:   baseline={resumen_base['dice_mean']:.4f}  v1={resumen_v1['dice_mean']:.4f}")
    print(f"clDice: baseline={resumen_base['cldice_mean']:.4f}  v1={resumen_v1['cldice_mean']:.4f}")
    print(f"topo_prec: baseline={resumen_base['topo_precision_mean']:.4f}  v1={resumen_v1['topo_precision_mean']:.4f}")
    print(f"topo_sens: baseline={resumen_base['topo_sensitivity_mean']:.4f}  v1={resumen_v1['topo_sensitivity_mean']:.4f}")
    print(f"n_componentes: baseline={resumen_base['n_componentes_mean']:.2f}  v1={resumen_v1['n_componentes_mean']:.2f}")
    print(f"HD95: baseline={resumen_base['hd95_crudo_mean']:.2f}  v1={resumen_v1['hd95_crudo_mean']:.2f}")

    print(f"\n--- Test pareado fold {fold} (Wilcoxon, n=200, delta>0 = v1 mejora) ---")
    efecto = {"fold": fold}
    for col, nombre, orientacion in METRICAS:
        a = df_base_s[col].values
        b = df_v1_s[col].values
        mask = ~(np.isnan(a) | np.isnan(b))
        w_stat, w_p = stats.wilcoxon(a[mask], b[mask])
        delta = orientacion * (b[mask] - a[mask])
        print(f"  {nombre}: Δmedio={delta.mean():+.4f} | Δmediano={np.median(delta):+.4f} | "
              f"Wilcoxon p={w_p:.4g} (n={mask.sum()})")
        efecto[f"{col}_delta_medio"] = delta.mean()
        efecto[f"{col}_delta_mediano"] = np.median(delta)
        efecto[f"{col}_wilcoxon_p"] = w_p
        efecto[f"{col}_n"] = mask.sum()

    media_cldice = comp["delta_cldice"].mean()
    std_cldice = comp["delta_cldice"].std()
    comp["z_cldice"] = (comp["delta_cldice"] - media_cldice) / std_cldice
    extremos = comp[comp["z_cldice"].abs() > 3]
    n_mejora = (comp["delta_cldice"] > 0).sum()
    n_empeora_comp = (comp["delta_n_componentes"] < 0).sum()
    print(f"\nclDice mejora con v1 en {n_mejora}/{len(comp)} casos ({100*n_mejora/len(comp):.1f}%)")
    print(f"v1 EMPEORA componentes (los aumenta) en {n_empeora_comp}/{len(comp)} casos "
          f"({100*n_empeora_comp/len(comp):.1f}%)")
    if len(extremos):
        print(f"{len(extremos)} caso(s) EXTREMO(S) en delta_cldice (>3 sigma):")
        print(extremos[["case_id","delta_cldice","z_cldice","delta_n_componentes"]].to_string(index=False))
    else:
        print("Ningun caso extremo (>3 sigma) en delta_cldice para este fold.")

    if imprimir_tabla_completa:
        pd.set_option("display.max_rows", None)
        cols_ver = ["case_id"] + [f"delta_{c[0]}" for c in METRICAS] + ["z_cldice"]
        print(f"\n=== Tabla completa fold {fold}, {len(comp)} casos, ordenada por delta_cldice ===")
        print(comp.sort_values("delta_cldice")[cols_ver].to_string(index=False))

    csv_comp = BASE / f"comparacion_caso_a_caso_fold{fold}_baseline_vs_v1.csv"
    comp.to_csv(csv_comp, index=False)
    print("Guardado:", csv_comp)
    return comp, efecto

print("Funcion comparar_fold lista (convenio de signo unico).")


Funcion comparar_fold lista (convenio de signo unico).


## 10. Procesado del fold 0 (recálculo) y de los folds 2, 3 y 4 (cálculo nuevo)

Es la etapa más costosa (descarga de ≈ 600 imágenes por fold y cálculo de métricas para 4 folds). Con el checkpoint por fold y trainer en Drive, una interrupción del runtime se resuelve volviendo a ejecutar esta celda, que retoma el proceso desde el punto en que quedó. Si el checkpoint procede de una versión antigua sin columnas topológicas (como el del fold 0), se detecta y se recalcula el fold completo.


In [10]:
resumenes = {}     # {fold: {trainer: resumen}}
dataframes = {}     # {fold: {trainer: df}}
comparaciones = {}  # {fold: df_comparacion}
efectos_por_fold = {}  # {fold: dict con delta medio + p por metrica}

for fold in FOLDS_A_PROCESAR:
    print(f"\n{'='*10} FOLD {fold} {'='*10}")
    local_val, local_gt = localizar_y_descargar_fold(fold)

    resumenes[fold] = {}
    dataframes[fold] = {}
    for trainer in (BASELINE_TRAINER, V1_TRAINER):
        df, resumen = evaluar_trainer_fold(fold, trainer, local_val, local_gt)
        resumenes[fold][trainer] = resumen
        dataframes[fold][trainer] = df

        etiqueta = "baseline" if trainer == BASELINE_TRAINER else "v1"
        csv_out = BASE / f"{etiqueta}_1000casos_fold{fold}_100epochs_metricas_por_caso.csv"
        df.to_csv(csv_out, index=False)
        print(f"  Guardado: {csv_out}")

    comp, efecto = comparar_fold(
        dataframes[fold][BASELINE_TRAINER], dataframes[fold][V1_TRAINER],
        resumenes[fold][BASELINE_TRAINER], resumenes[fold][V1_TRAINER], fold,
    )
    comparaciones[fold] = comp
    efectos_por_fold[fold] = efecto

print(f"\nFolds {FOLDS_A_PROCESAR} evaluados/recalculados.")



========== FOLD 0 ==========


  fold 0: descargando 400 predicciones + 200 GT (en paralelo, 8 workers)...


    50/200 procesados (0 fallos hasta ahora)
    100/200 procesados (0 fallos hasta ahora)
    150/200 procesados (0 fallos hasta ahora)


    200/200 procesados (0 fallos hasta ahora)
    Descarga paralela (8 workers): 200 archivos en 1.4 min (0.41 s/archivo)


    50/200 procesados (0 fallos hasta ahora)
    100/200 procesados (0 fallos hasta ahora)
    150/200 procesados (0 fallos hasta ahora)


...

    50/200 procesados (0 fallos hasta ahora)
    100/200 procesados (0 fallos hasta ahora)
    150/200 procesados (0 fallos hasta ahora)


    200/200 procesados (0 fallos hasta ahora)
    Descarga paralela (8 workers): 200 archivos en 1.3 min (0.40 s/archivo)


    50/200 procesados (0 fallos hasta ahora)
    100/200 procesados (0 fallos hasta ahora)
    150/200 procesados (0 fallos hasta ahora)


    200/200 procesados (0 fallos hasta ahora)
    Descarga paralela (8 workers): 200 archivos en 1.3 min (0.40 s/archivo)


    50/200 procesados (0 fallos hasta ahora)
    100/200 procesados (0 fallos hasta ahora)
    150/200 procesados (0 fallos hasta ahora)
    200/200 procesados (0 fallos hasta ahora)
    Descarga paralela (8 workers): 200 archivos en 7.3 min (2.20 s/archivo)
    [fold 4 / nnUNetTrainer_100epochs] calculando 200 casos pendientes con 2 procesos en paralelo...
    [fold 4 / nnUNetTrainer_100epochs] 10/200 procesados -- checkpoint guardado
    [fold 4 / nnUNetTrainer_100epochs] 20/200 procesados -- checkpoint guardado
    [fold 4 / nnUNetTrainer_100epochs] 30/200 procesados -- checkpoint guardado
    [fold 4 / nnUNetTrainer_100epochs] 40/200 procesados -- checkpoint guardado
    [fold 4 / nnUNetTrainer_100epochs] 50/200 procesados -- checkpoint guardado
    [fold 4 / nnUNetTrainer_100epochs] 60/200 procesados -- checkpoint guardado
    [fold 4 / nnUNetTrainer_100epochs] 70/200 procesados -- checkpoint guardado
    [fold 4 / nnUNetTrainer_100epochs] 80/200 procesados -- checkpoint guardado


## 11. Carga del fold 1 (completo, sin recálculo) y verificación de consistencia global

El fold 1 se evaluó en el notebook 32 y ya contiene las seis métricas (incluidas `topo_precision` y `topo_sensitivity`), por lo que solo se carga y se procesa con `comparar_fold` para obtener su fila en las tablas B y C, sin volver a descargar predicciones de Drive.


In [11]:
df_base_f1 = pd.read_csv(BASE / f"baseline_1000casos_fold{FOLD_YA_COMPLETO}_100epochs_metricas_por_caso.csv")
df_v1_f1 = pd.read_csv(BASE / f"v1_1000casos_fold{FOLD_YA_COMPLETO}_100epochs_metricas_por_caso.csv")
assert {"topo_precision", "topo_sensitivity"}.issubset(df_base_f1.columns), \
    f"El CSV de fold {FOLD_YA_COMPLETO} no tiene columnas topologicas -- revisa que sea el correcto."

resumen_base_f1 = {
    "dice_mean": df_base_f1["dice"].mean(), "cldice_mean": df_base_f1["cldice"].mean(),
    "topo_precision_mean": df_base_f1["topo_precision"].mean(), "topo_sensitivity_mean": df_base_f1["topo_sensitivity"].mean(),
    "n_componentes_mean": df_base_f1["n_componentes"].mean(), "hd95_crudo_mean": df_base_f1["hd95_crudo_mm"].mean(),
}
resumen_v1_f1 = {
    "dice_mean": df_v1_f1["dice"].mean(), "cldice_mean": df_v1_f1["cldice"].mean(),
    "topo_precision_mean": df_v1_f1["topo_precision"].mean(), "topo_sensitivity_mean": df_v1_f1["topo_sensitivity"].mean(),
    "n_componentes_mean": df_v1_f1["n_componentes"].mean(), "hd95_crudo_mean": df_v1_f1["hd95_crudo_mm"].mean(),
}

comp_f1, efecto_f1 = comparar_fold(df_base_f1, df_v1_f1, resumen_base_f1, resumen_v1_f1, FOLD_YA_COMPLETO)
comparaciones[FOLD_YA_COMPLETO] = comp_f1
efectos_por_fold[FOLD_YA_COMPLETO] = efecto_f1

dataframes[FOLD_YA_COMPLETO] = {BASELINE_TRAINER: df_base_f1, V1_TRAINER: df_v1_f1}
print("\nFold 1 cargado e incorporado a las tablas B y C.")



=== fold 1: resumen ===
Dice:   baseline=0.7541  v1=0.7666
clDice: baseline=0.8208  v1=0.8348
topo_prec: baseline=0.7651  v1=0.7907
topo_sens: baseline=0.8979  v1=0.8939
n_componentes: baseline=15.98  v1=13.64
HD95: baseline=22.56  v1=18.70

--- Test pareado fold 1 (Wilcoxon, n=200, delta>0 = v1 mejora) ---
  Dice: Δmedio=+0.0125 | Δmediano=+0.0090 | Wilcoxon p=2.773e-21 (n=200)
  clDice: Δmedio=+0.0140 | Δmediano=+0.0100 | Wilcoxon p=3.219e-18 (n=200)
  Topological precision: Δmedio=+0.0256 | Δmediano=+0.0216 | Wilcoxon p=1.591e-24 (n=200)
  Topological sensitivity: Δmedio=-0.0040 | Δmediano=-0.0023 | Wilcoxon p=8.262e-05 (n=200)
  n_componentes: Δmedio=+2.3450 | Δmediano=+2.0000 | Wilcoxon p=2.721e-16 (n=200)
  HD95: Δmedio=+3.8522 | Δmediano=+1.7717 | Wilcoxon p=1.023e-24 (n=200)

clDice mejora con v1 en 152/200 casos (76.0%)
v1 EMPEORA componentes (los aumenta) en 41/200 casos (20.5%)
3 caso(s) EXTREMO(S) en delta_cldice (>3 sigma):
      case_id  delta_cldice  z_cldice  delta_n_c

## 12. Unión de los 5 folds en un único pool de 1000 casos

Los cinco folds comparten exactamente las mismas columnas (incluidas las topológicas), por lo que no es necesario ningún tratamiento especial de valores NaN.


In [12]:
dfs_base_todos = []
dfs_v1_todos = []
for fold in TODOS_LOS_FOLDS:
    db = dataframes[fold][BASELINE_TRAINER].copy(); db["fold"] = fold
    dv = dataframes[fold][V1_TRAINER].copy(); dv["fold"] = fold
    dfs_base_todos.append(db)
    dfs_v1_todos.append(dv)
    print(f"fold {fold}: baseline {len(db)} casos, v1 {len(dv)} casos, "
          f"columnas topologicas presentes: {'topo_precision' in db.columns}")

df_base_todos = pd.concat(dfs_base_todos, ignore_index=True)
df_v1_todos = pd.concat(dfs_v1_todos, ignore_index=True)

print(f"\nTotal baseline: {len(df_base_todos)} casos, total v1: {len(df_v1_todos)} casos")
assert len(df_base_todos) == 1000 and len(df_v1_todos) == 1000, \
    "No hay 1000 casos en total -- revisa que los 5 folds tengan 200 cada uno."
assert df_base_todos["case_id"].nunique() == 1000, "Hay case_id duplicados en baseline."
assert df_v1_todos["case_id"].nunique() == 1000, "Hay case_id duplicados en v1."
assert set(df_base_todos["case_id"]) == set(df_v1_todos["case_id"]), \
    "Los case_id de baseline y v1 no coinciden -- revisa los splits."
for col, nombre, _ in METRICAS:
    n_validos = df_base_todos[col].notna().sum()
    assert n_validos == 1000, f"{nombre}: solo {n_validos}/1000 casos validos (deberian ser 1000)."

print("Verificado: 1000 casos unicos en cada modelo, las 6 metricas completas en los 1000 casos.")


fold 0: baseline 200 casos, v1 200 casos, columnas topologicas presentes: True
fold 1: baseline 200 casos, v1 200 casos, columnas topologicas presentes: True
fold 2: baseline 200 casos, v1 200 casos, columnas topologicas presentes: True
fold 3: baseline 200 casos, v1 200 casos, columnas topologicas presentes: True
fold 4: baseline 200 casos, v1 200 casos, columnas topologicas presentes: True

Total baseline: 1000 casos, total v1: 1000 casos
Verificado: 1000 casos unicos en cada modelo, las 6 metricas completas en los 1000 casos.


## 13. (A) Resultado pooled sobre las 1000 imágenes

Resume la magnitud global del efecto, **sin presentarse como el resultado de la validación cruzada**: las predicciones proceden de cinco modelos distintos entrenados con particiones solapadas, y la cuestión relevante es si el efecto se reproduce entre folds (secciones 14 y 15). Convenio de signo: `delta > 0` indica que v1 mejora, para las seis métricas.


In [13]:
df_base_s = df_base_todos.sort_values("case_id").reset_index(drop=True)
df_v1_s = df_v1_todos.sort_values("case_id").reset_index(drop=True)
assert list(df_base_s["case_id"]) == list(df_v1_s["case_id"])

print("=== (A) Pooled, 5-fold CV completo, n=1000 (delta>0 = v1 mejora) ===")
filas_pool = []
for col, nombre, orientacion in METRICAS:
    a = df_base_s[col].values
    b = df_v1_s[col].values
    mask = ~(np.isnan(a) | np.isnan(b))
    a_m, b_m = a[mask], b[mask]
    delta = orientacion * (b_m - a_m)
    w_stat, w_p = stats.wilcoxon(a_m, b_m)
    t_stat, t_p = stats.ttest_rel(a_m, b_m)
    print(f"{nombre}: n={mask.sum()} | Δmedio={delta.mean():+.4f} | Δmediano={np.median(delta):+.4f} | "
          f"Wilcoxon p={w_p:.4g} | ttest_rel p={t_p:.4g}")
    filas_pool.append({
        "metrica": nombre, "n": mask.sum(),
        "media_baseline": a_m.mean(), "media_v1": b_m.mean(),
        "delta_medio": delta.mean(), "delta_mediano": np.median(delta),
        "wilcoxon_p": w_p, "ttest_rel_p": t_p,
    })

df_pool = pd.DataFrame(filas_pool)
csv_pool = BASE / "A_tests_pareados_POOL_5fold_baseline_vs_v1.csv"
df_pool.to_csv(csv_pool, index=False)
print("\nGuardado:", csv_pool)


=== (A) Pooled, 5-fold CV completo, n=1000 (delta>0 = v1 mejora) ===
Dice: n=1000 | Δmedio=-0.0021 | Δmediano=-0.0023 | Wilcoxon p=2.367e-08 | ttest_rel p=7.646e-05
clDice: n=1000 | Δmedio=-0.0005 | Δmediano=-0.0013 | Wilcoxon p=0.07006 | ttest_rel p=0.4111
Topological precision: n=1000 | Δmedio=-0.0070 | Δmediano=-0.0090 | Wilcoxon p=2.904e-15 | ttest_rel p=1.382e-11
Topological sensitivity: n=1000 | Δmedio=+0.0085 | Δmediano=+0.0067 | Wilcoxon p=4.413e-62 | ttest_rel p=6.899e-57
n_componentes: n=1000 | Δmedio=+0.7310 | Δmediano=+1.0000 | Wilcoxon p=8.868e-11 | ttest_rel p=3.853e-10
HD95: n=1000 | Δmedio=-0.5349 | Δmediano=-0.1561 | Wilcoxon p=0.0007403 | ttest_rel p=0.009873

Guardado: /content/drive/MyDrive/TFM_nnUNet/A_tests_pareados_POOL_5fold_baseline_vs_v1.csv


## 14. (B) Resultado independiente de cada fold

Tabla con una fila por fold: delta medio (convenio «positivo = mejora») y p-valor de Wilcoxon de cada métrica, calculados **únicamente con los 200 casos de ese fold**.


In [14]:
tabla_B = pd.DataFrame([efectos_por_fold[f] for f in TODOS_LOS_FOLDS]).sort_values("fold").reset_index(drop=True)

cols_delta = [f"{c[0]}_delta_medio" for c in METRICAS]
cols_p = [f"{c[0]}_wilcoxon_p" for c in METRICAS]
print("=== (B) Delta medio (v1 mejora si >0) por fold ===")
print(tabla_B[["fold"] + cols_delta].to_string(index=False))
print("\n=== (B) p-valor Wilcoxon por fold ===")
print(tabla_B[["fold"] + cols_p].to_string(index=False))

csv_B = BASE / "B_resultado_por_fold_baseline_vs_v1.csv"
tabla_B.to_csv(csv_B, index=False)
print("\nGuardado:", csv_B)


=== (B) Delta medio (v1 mejora si >0) por fold ===
 fold  dice_delta_medio  cldice_delta_medio  topo_precision_delta_medio  topo_sensitivity_delta_medio  n_componentes_delta_medio  hd95_crudo_mm_delta_medio
    0         -0.004093           -0.002362                   -0.016007                      0.016534                     -0.215                  -1.275166
    1          0.012461            0.013970                    0.025627                     -0.003984                      2.345                   3.852174
    2         -0.001328           -0.004412                   -0.016897                      0.013397                      0.760                  -0.148533
    3         -0.003926            0.001172                   -0.003971                      0.008654                      1.275                  -0.491663
    4         -0.013734           -0.010943                   -0.023838                      0.008016                     -0.510                  -4.611406

=== (B) p-va

## 15. (C) Consistencia del efecto entre folds

Responde a la cuestión de la robustez: no se trata de la media ± SD de la métrica en sí (visible en la tabla B), sino de la **media ± SD del delta (v1 - baseline) a través de los cinco folds**. Una SD grande respecto a la media indica que el efecto de λ = 0.2 no es estable entre folds, con independencia de que el resultado pooled sea significativo.


In [15]:
filas_C = []
for col, nombre, orientacion in METRICAS:
    deltas_5folds = tabla_B[f"{col}_delta_medio"].values
    filas_C.append({
        "metrica": nombre,
        "delta_medio_entre_folds": deltas_5folds.mean(),
        "delta_std_entre_folds": deltas_5folds.std(),
        "delta_min": deltas_5folds.min(),
        "delta_max": deltas_5folds.max(),
        "n_folds_con_v1_mejor": int((deltas_5folds > 0).sum()),
    })
tabla_C = pd.DataFrame(filas_C)

print("=== (C) Consistencia del efecto v1-baseline entre los 5 folds ===")
print(tabla_C.to_string(index=False))
print("\n'n_folds_con_v1_mejor' de 5 dice en cuantos folds v1 gana en esa metrica "
      "(5/5 = consistente en todos, 3/5 o menos = efecto poco fiable).")

csv_C = BASE / "C_consistencia_efecto_entre_folds_baseline_vs_v1.csv"
tabla_C.to_csv(csv_C, index=False)
print("\nGuardado:", csv_C)


=== (C) Consistencia del efecto v1-baseline entre los 5 folds ===
                metrica  delta_medio_entre_folds  delta_std_entre_folds  delta_min  delta_max  n_folds_con_v1_mejor
                   Dice                -0.002124               0.008429  -0.013734   0.012461                     1
                 clDice                -0.000515               0.008246  -0.010943   0.013970                     2
  Topological precision                -0.007017               0.017529  -0.023838   0.025627                     1
Topological sensitivity                 0.008523               0.006995  -0.003984   0.016534                     4
          n_componentes                 0.731000               1.033128  -0.510000   2.345000                     3
                   HD95                -0.534919               2.704199  -4.611406   3.852174                     1

'n_folds_con_v1_mejor' de 5 dice en cuantos folds v1 gana en esa metrica (5/5 = consistente en todos, 3/5 o menos = efect

## Resumen de salidas

El notebook proporciona tres elementos complementarios para evaluar si soft-clDice (λ = 0.2) mejora la continuidad vascular de forma fiable:

- **A** (`A_tests_pareados_POOL_5fold_baseline_vs_v1.csv`): magnitud del efecto con la máxima potencia estadística (n = 1000), aunque mezclando cinco modelos distintos.
- **B** (`B_resultado_por_fold_baseline_vs_v1.csv`): resultado de cada fold por separado, para identificar folds con comportamiento distinto (como ocurrió entre el fold 0 y el fold 1).
- **C** (`C_consistencia_efecto_entre_folds_baseline_vs_v1.csv`): reproducibilidad del efecto entre folds (SD pequeña y mayoría de folds con v1 mejor) o fragilidad (SD grande y resultados dispersos).

Si el runtime se interrumpe durante la sección 10, se vuelve a ejecutar desde la sección 1: los checkpoints por fold y trainer en Drive evitan recalcular lo ya hecho, y la descarga por fold también es reanudable (verifica por tamaño).
